# 🖼️ Урок 15 — Свёрточные сети CNN (материалы преподавателя)

Каждый блок: что делаем · зачем · что ожидаем.

> 🎯 Цель: понять свёртку-фильтр, построить CNN и сравнить с обычной сетью, познакомиться с transfer learning.

## Блок 1 · Что делает фильтр (свёртка краёв)
**Что делаем:** двигаем маленький фильтр по картинке.
**Что ожидаем:** он подсветит границы объекта — это карта признаков.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.signal import convolve2d
img = np.zeros((20,20)); img[5:15, 5:15] = 1     # белый квадрат
фильтр = np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])   # реагирует на края
карта = convolve2d(img, фильтр, mode='valid')
plt.subplot(1,2,1); plt.imshow(img, cmap='gray'); plt.title('исходная')
plt.subplot(1,2,2); plt.imshow(карта, cmap='gray'); plt.title('края'); plt.show()

## Блок 2 · CNN на CIFAR-10
**Что ожидаем:** сеть учится узнавать объекты (самолёт, кошка, машина …).

In [ ]:
from tensorflow import keras
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
X_train, X_test = X_train/255.0, X_test/255.0
cnn = keras.Sequential([
    keras.Input(shape=(32,32,3)),
    keras.layers.Conv2D(32,(3,3),activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64,(3,3),activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64,activation='relu'),
    keras.layers.Dense(10,activation='softmax'),
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.fit(X_train, y_train, epochs=5, validation_split=0.1)
print(f'CNN точность: {cnn.evaluate(X_test, y_test, verbose=0)[1]:.1%}')

## Блок 3 · Сравнение с полносвязной сетью
**Что ожидаем:** обычная сеть заметно хуже на картинках.

In [ ]:
dense = keras.Sequential([keras.Input(shape=(32,32,3)), keras.layers.Flatten(),
    keras.layers.Dense(128,activation='relu'), keras.layers.Dense(10,activation='softmax')])
dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
dense.fit(X_train, y_train, epochs=5, validation_split=0.1, verbose=0)
print(f'Обычная сеть: {dense.evaluate(X_test, y_test, verbose=0)[1]:.1%}')

## Блок 4 · Transfer learning (MobileNetV2)
**Что делаем:** берём готовую сеть и доучиваем только 'голову'.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights='imagenet')
base.trainable = False
model = keras.Sequential([base, keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(3, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

---
**Итог.** Свёртка = скользящий фильтр, ищущий узоры. CNN учитывает соседство пикселей и точнее обычной сети. Transfer learning экономит время и данные.